# 👑 GeoMAS - Research Master Dashboard

Questo notebook è progettato per supportare l'analisi dei dati per la Tesi Magistrale.
Organizzato per rispondere alle 4 Research Questions (RQ) principali, mantenendo tutti i monitor di telemetria standard.

### Indice dei Capitoli:
- **Capitolo 0**: Telemetria Standard (Preserved Legacy)
- **Capitolo 1 (RQ1)**: Scenario Analysis (Baseline vs Scenario)
- **Capitolo 2 (RQ2)**: Etica e Analisi Controfattuale (XAI Comparison)
- **Capitolo 3 (RQ3)**: Onestà Strategica e Moral Washing
- **Capitolo 4 (RQ4)**: Equilibri Socio-Politici e Convergenza
- **Capitolo 5**: Monitoraggi Tecnici (Risorse ed Economia)

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.lines as mlines
import numpy as np

# Configurazione Stile Master
sns.set_theme(style="darkgrid", context="talk")
plt.rcParams['figure.figsize'] = (14, 7)
COLORS = {'defense': '#e74c3c', 'foreign': '#3498db', 'economy': '#f1c40f', 'satisfaction': '#2ecc71'}

DB_PATH = 'simulation_metrics.duckdb'

def query_db(query, params=None):
    if params:
        params = [p.item() if hasattr(p, 'item') else p for p in params]
    with duckdb.connect(DB_PATH, read_only=True) as conn:
        return conn.execute(query, params or []).df()

## ⚙️ Selezione Dati
Configura gli ID delle simulazioni per il confronto (RQ1/RQ2).

In [ ]:
simulations = query_db("SELECT DISTINCT simulation_id FROM metrics_global ORDER BY simulation_id")
print("Simulazioni Disponibili:")
display(simulations)

# Seleziona gli ID
TARGET_ID = simulations['simulation_id'].iloc[-1] if not simulations.empty else None
BASELINE_ID = simulations['simulation_id'].iloc[-2] if len(simulations) > 1 else None

print(f"Target Sim (Current/Counterfactual): {TARGET_ID}")
print(f"Baseline Sim (Start/Reference): {BASELINE_ID}")

def load_sim_data(sim_id):
    if not sim_id: return None, None, None
    g = query_db("SELECT * FROM metrics_global WHERE simulation_id = ? ORDER BY turn", [sim_id])
    n = query_db("SELECT * FROM metrics_nation WHERE simulation_id = ? ORDER BY turn, nation_id", [sim_id])
    t = query_db("SELECT * FROM metrics_trust WHERE simulation_id = ? ORDER BY turn", [sim_id])
    return g, n, t

global_df, nation_df, trust_df = load_sim_data(TARGET_ID)

## 📊 Capitolo 0: Telemetria Standard
Visualizzazioni classiche per il monitoraggio rapido.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 18))

# 0.1 Performance Medie
sns.lineplot(data=global_df, x='turn', y='global_deception_avg', ax=axes[0], label='Deception Media', color='red')
sns.lineplot(data=global_df, x='turn', y='global_coherence_avg', ax=axes[0], label='Coherence Media', color='green')
axes[0].set_title("Qualità degli Agenti (Media Globale)")
axes[0].set_ylim(-0.1, 1.1)

# 0.2 Guns vs Butter
ax_butter = axes[1]
ax_guns = ax_butter.twinx()
sns.lineplot(data=global_df, x='turn', y='global_trade_volume', ax=ax_butter, color='blue', label='Trade')
sns.lineplot(data=global_df, x='turn', y='units_created', ax=ax_guns, color='orange', label='Military')
ax_butter.set_title("Guns vs Butter (Global Allocations)")

# 0.3 Power Projection
sns.lineplot(data=nation_df, x='turn', y='power_projection', hue='nation_id', ax=axes[2])
axes[2].set_title("Evoluzione del Potere (Power Projection)")

plt.tight_layout()
plt.show()

## ⚖️ Capitolo 1 & 2 (RQ1 & RQ2): Scenario and Counterfactual Analysis
Confronto tra run per valutare l'impatto degli scenari e le scelte etiche (XAI).

In [ ]:
if BASELINE_ID:
    g_base, n_base, t_base = load_sim_data(BASELINE_ID)
    comp_df = pd.concat([global_df.assign(sim='Target (Scenario/XAI)'), g_base.assign(sim='Baseline')])
    
    fig, ax = plt.subplots(1, 2, figsize=(18, 6))
    # RQ1: Impact on Coherence/Deception
    sns.lineplot(data=comp_df, x='turn', y='global_deception_avg', hue='sim', ax=ax[0])
    ax[0].set_title("RQ1: Impatto sulla Deception Globale")
    
    # RQ2: Impact on Resources (Ethics Index)
    sns.lineplot(data=comp_df, x='turn', y='global_trade_volume', hue='sim', ax=ax[1])
    ax[1].set_title("RQ2: Impatto sul Commercio (Indicatore Etico)")
    
    plt.show()
else:
    print("Confronto non disponibile: seleziona una Baseline ID.")

## 🛡️ Capitolo 3 (RQ3): Onestà Strategica e Moral Washing
Analisi della dicotomia tra l'inganno nei diversi domini ministeriali (Difesa vs Esteri).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 3.1 Domain Deception (Strategic Inconsistency)
gap_df = nation_df.groupby('turn')[['deception_foreign', 'deception_defense']].mean().reset_index()
axes[0].plot(gap_df['turn'], gap_df['deception_defense'], label='Deception: Defense Domain', color=COLORS['defense'], linewidth=3)
axes[0].plot(gap_df['turn'], gap_df['deception_foreign'], label='Deception: Foreign Domain', color=COLORS['foreign'], linewidth=3)
axes[0].set_title("Deception per Dominio (Mismatch tra Intento Pubblico e Privato)")
axes[0].set_ylabel("Deception Score (Higher = More Deceptive)")
axes[0].legend()

# 3.2 Deception Score Individuata
sns.lineplot(data=nation_df, x='turn', y='deception_overall', hue='nation_id', ax=axes[1])
axes[1].set_title("Deception Score Totale per Nazione")
axes[1].set_ylim(-0.05, 1.05)

plt.show()

## 🌐 Capitolo 4 (RQ4): Equilibri Socio-Politici
Analisi della convergenza del network e della stabilità (Satisfaction).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(22, 10))

# 4.1 Network Map (Ultimo Turno) - Semplificata senza doppie frecce
last_t = nation_df['turn'].max()
t_now = trust_df[trust_df['turn'] == last_t]

if not t_now.empty:
    # Usiamo Graph() invece di DiGraph() per evitare le frecce e avere un solo arco
    G = nx.Graph()
    COLORS_REL = {'MUTUAL_DEFENSE': 'green', 'NON_AGGRESSION': 'yellow', 'WAR': 'red', 'PEACE': 'gray'}

    # Priorità degli stati (se A->B è PEACE ma B->A è WAR, mostriamo WAR)
    REL_PRIORITY = {'WAR': 3, 'MUTUAL_DEFENSE': 2, 'NON_AGGRESSION': 1, 'PEACE': 0}

    for _, row in t_now.iterrows():
        u, v = row['observer_id'], row['target_id']
        if u == v: continue

        # Ordiniamo i nodi per trattare (A,B) e (B,A) come lo stesso arco
        edge = tuple(sorted([u, v]))
        rel = row['relationship_state']
        color = COLORS_REL.get(rel, 'gray')

        # Aggiungiamo l'arco o aggiorniamolo solo se la nuova relazione ha priorità maggiore
        if G.has_edge(*edge):
            current_color = G[edge[0]][edge[1]]['color']
            # Trova lo stato corrispondente al colore attuale per confrontare la priorità
            current_rel = next((k for k, v in COLORS_REL.items() if v == current_color), 'PEACE')
            if REL_PRIORITY.get(rel, 0) > REL_PRIORITY.get(current_rel, 0):
                G.add_edge(*edge, color=color)
        else:
            G.add_edge(*edge, color=color)

    pos = nx.spring_layout(G, seed=42)
    # Rimosso connectionstyle per avere linee dritte
    nx.draw(G, pos,
            with_labels=True,
            node_color='lightgray',
            edge_color=[G[u][v]['color'] for u, v in G.edges()],
            ax=ax[0],
            node_size=3000,
            width=2)
    ax[0].set_title("Stato delle Relazioni Globali (Sintesi)")

# 4.2 Satisfaction Trends (Nations)
sns.lineplot(data=nation_df, x='turn', y='public_satisfaction', hue='nation_id', ax=ax[1], linewidth=2)
ax[1].set_title("Stabilità delle Nazioni (Soddisfazione Pubblica)")
ax[1].axhline(50, color='gray', linestyle='--')

plt.show()

## 🍎 Capitolo 5: Monitoraggio Risorse (Health Check)
Utilizza questo capitolo per diagnosticare problemi di scarsità o bilanciamento.

In [ ]:
res_list = ['budget', 'food', 'energy', 'materials']
fig, ax = plt.subplots(2, 2, figsize=(18, 14))
axes = ax.flatten()

for i, res in enumerate(res_list):
    sns.lineplot(data=nation_df, x='turn', y=res, hue='nation_id', ax=axes[i], legend=(i==0))
    axes[i].set_title(f"Scorte di {res.capitalize()}")
    axes[i].axhline(50, color='red', linestyle='--', alpha=0.3)
    
plt.tight_layout()
plt.show()